# IT Interview Assistant: KNN Next-Question Recommender
```
CSV → clean → TF-IDF vectors → score answer (cosine) → KNN picks next question → save model for API
```

## 1. Imports and config

In [ ]:
import pandas as pd, numpy as np, joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

CSV_PATH   = "Software_Questions.csv"
Q_COL      = "Question"      # change to match your CSV
A_COL      = "Answer"        # change to match your CSV
CAREER_COL = "Category"      # used as the career; if missing, "General" is used
DIFF_COL   = "Difficulty"    # optional; derived automatically if missing
PASS_SCORE = 0.45            # similarity >= this counts as a correct answer
K          = 5

## 2. Load data (first 20 + last 15 rows)

In [ ]:
raw = pd.read_csv(CSV_PATH)
df = pd.concat([raw.head(20), raw.tail(15)]).drop_duplicates().reset_index(drop=True)
df = df.rename(columns={Q_COL: "question", A_COL: "answer"})
df["career"] = df[CAREER_COL].astype(str) if CAREER_COL in df.columns else "General"

# difficulty 1-3: use the column if present, else split by reference-answer length
if DIFF_COL in df.columns:
    m = {"easy": 1, "medium": 2, "hard": 3}
    df["difficulty"] = df[DIFF_COL].astype(str).str.lower().map(m).fillna(2).astype(int)
else:
    df["difficulty"] = pd.qcut(df["answer"].str.len().rank(method="first"), 3, labels=[1, 2, 3]).astype(int)

df = df[["question", "answer", "career", "difficulty"]].dropna().reset_index(drop=True)
df["id"] = df.index
df.head()

## 3. Vectorise (TF-IDF)
Each row is represented by its question + answer text, so similar topics sit close together.

In [ ]:
tfidf = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), sublinear_tf=True)
X = tfidf.fit_transform(df["question"] + " " + df["answer"])
X.shape

## 4. Score a user's answer
Cosine similarity between the user's answer and the reference answer, from 0 to 1.

In [ ]:
def score_answer(user_answer, qid):
    u = tfidf.transform([user_answer])
    r = tfidf.transform([df.loc[qid, "answer"]])
    return float(cosine_similarity(u, r)[0, 0])

def is_correct(score): return score >= PASS_SCORE

## 5. KNN model

In [ ]:
knn = NearestNeighbors(n_neighbors=min(K + 15, len(df)), metric="cosine", algorithm="brute")
knn.fit(X)

## 6. Next-question logic
```
score high  → nearest questions of SAME or HIGHER difficulty (progress)
score low   → nearest questions of SAME or LOWER difficulty (reinforce)
filters: same career, not already asked
```

In [ ]:
def first_question(career, rng=np.random.default_rng()):
    """Cold start: random medium-difficulty question for the career."""
    pool = df[df.career == career]
    mid = pool[pool.difficulty == 2]
    pool = mid if len(mid) else pool
    return int(rng.choice(pool["id"].values))

def next_question(current_id, score, asked, career):
    dist, idx = knn.kneighbors(X[current_id], n_neighbors=len(df))
    cur_diff = df.loc[current_id, "difficulty"]
    up = is_correct(score)
    cands = []
    for d, i in zip(dist[0], idx[0]):
        if i in asked or df.loc[i, "career"] != career:
            continue
        diff = df.loc[i, "difficulty"]
        if (up and diff >= cur_diff) or (not up and diff <= cur_diff):
            cands.append((d, i))
    if not cands:  # fallback: nearest unasked question in the career
        cands = [(d, i) for d, i in zip(dist[0], idx[0])
                 if i not in asked and df.loc[i, "career"] == career]
    return int(cands[0][1]) if cands else None  # None = test finished

## 7. Simulate an interview

In [ ]:
career = df["career"].iloc[0]
asked, log = [], []
qid = first_question(career)

for _ in range(5):
    asked.append(qid)
    print("Q:", df.loc[qid, "question"])
    # simulated answers: alternate good (reference text) and weak
    user_ans = df.loc[qid, "answer"] if len(asked) % 2 else "I am not sure"
    s = score_answer(user_ans, qid)
    log.append({"id": qid, "question": df.loc[qid, "question"], "user_answer": user_ans,
                "score": round(s, 2), "correct": is_correct(s), "difficulty": int(df.loc[qid, "difficulty"])})
    print(f"   score={s:.2f} correct={is_correct(s)}")
    qid = next_question(qid, s, asked, career)
    if qid is None: break

pd.DataFrame(log)

## 8. Evaluate (quick sanity check)
How often is the reference answer scored as correct, and a wrong answer as incorrect?

In [ ]:
ref  = np.mean([is_correct(score_answer(df.loc[i, "answer"], i)) for i in df.id])
wrong = np.mean([is_correct(score_answer(df.loc[(i + 1) % len(df), "answer"], i)) for i in df.id])
print(f"Reference answers accepted: {ref:.0%}")
print(f"Another question's answer wrongly accepted: {wrong:.0%}")

## 9. Save for the API
The session log (`log`) is what the RAG feedback step will use after the test.

In [ ]:
joblib.dump({"df": df, "tfidf": tfidf, "X": X, "knn": knn, "pass_score": PASS_SCORE}, "knn_model.joblib")
print("saved knn_model.joblib")